In [1]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from transformers import CLIPProcessor, CLIPModel
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os
import shutil
import numpy as np
import pandas as pd
import plotly.express as px
import glob
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm

# --- EINSTELLUNGEN ---
# INPUT_DIR = '../DSAI_shrinked_images'
INPUT_DIR = '../../dataset'
OUTPUT_BASE = '../DSAI_Ultimate_Output'
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 32

class AI_Mastermind:
    def __init__(self, input_dir):
        self.input_dir = input_dir
        self.image_paths = []
        # Unterstützte Formate
        for ext in ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.PNG"]:
            self.image_paths.extend(glob.glob(os.path.join(input_dir, ext)))
        
        print(f"[{len(self.image_paths)} Bilder geladen]")
        self.features = None
        self.embeddings_clip = None
        self.paths_sorted = sorted(self.image_paths) # Konsistente Reihenfolge
        
    def _load_resnet(self):
        print(">>> Lade ResNet50 (für Struktur & Duplikate)...")
        model = models.resnet50(weights='DEFAULT')
        model.fc = nn.Identity()
        model.to(DEVICE)
        model.eval()
        return model

    def _load_clip(self):
        print(">>> Lade CLIP (für Text-Sortierung)...")
        model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEVICE)
        processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
        return model, processor

    def extract_visual_features(self):
        """Extrahiert Features mit ResNet für Clustering, 3D-Viz und Duplikate"""
        if self.features is not None: return # Schon erledigt

        model = self._load_resnet()
        transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

        feats = []
        print(">>> Scanne Bilder (ResNet)...")
        # Wir machen das ohne DataLoader für simplen Code, aber mit Batch-Logik wäre es noch schneller
        with torch.no_grad():
            for i in tqdm(range(0, len(self.paths_sorted), BATCH_SIZE)):
                batch_paths = self.paths_sorted[i:i+BATCH_SIZE]
                batch_imgs = []
                valid_indices = []
                
                for idx, p in enumerate(batch_paths):
                    try:
                        img = Image.open(p).convert("RGB")
                        batch_imgs.append(transform(img))
                        valid_indices.append(idx)
                    except: pass
                
                if not batch_imgs: continue
                
                tensor = torch.stack(batch_imgs).to(DEVICE)
                out = model(tensor)
                feats.append(out.cpu().numpy())
        
        self.features = np.concatenate(feats, axis=0)
        print(f">>> Features fertig: {self.features.shape}")

    def sort_by_text_categories(self, categories):
        """Sortiert Bilder basierend auf DEINEN Begriffen (CLIP)"""
        model, processor = self._load_clip()
        
        print(f">>> Sortiere nach: {categories}")
        dest_dir = os.path.join(OUTPUT_BASE, "Sorted_by_Text")
        if os.path.exists(dest_dir): shutil.rmtree(dest_dir)
        
        # Batch-Verarbeitung für CLIP
        with torch.no_grad():
            for i in tqdm(range(0, len(self.paths_sorted), BATCH_SIZE)):
                batch_paths = self.paths_sorted[i:i+BATCH_SIZE]
                images = []
                valid_paths = []
                
                for p in batch_paths:
                    try:
                        images.append(Image.open(p).convert("RGB"))
                        valid_paths.append(p)
                    except: continue
                
                if not images: continue
                
                inputs = processor(text=categories, images=images, return_tensors="pt", padding=True).to(DEVICE)
                outputs = model(**inputs)
                
                # Wahrscheinlichkeiten: Welches Bild passt zu welchem Text?
                probs = outputs.logits_per_image.softmax(dim=1)
                
                for idx, prob_vector in enumerate(probs):
                    best_cat_idx = prob_vector.argmax().item()
                    confidence = prob_vector[best_cat_idx].item()
                    category = categories[best_cat_idx]
                    path = valid_paths[idx]
                    
                    # Nur sortieren wenn halbwegs sicher (>20%)
                    folder = category if confidence > 0.2 else "Unsicher"
                    
                    target = os.path.join(dest_dir, folder)
                    os.makedirs(target, exist_ok=True)
                    shutil.copy2(path, os.path.join(target, os.path.basename(path)))
        
        print(f"Fertig! Ergebnisse in {dest_dir}")

    def find_duplicates(self, threshold=0.02):
        """Findet exakte oder fast identische Bilder"""
        self.extract_visual_features() # ResNet Features nutzen
        
        print(">>> Suche Zwillinge...")
        # Berechne Ähnlichkeit aller Bilder zueinander
        sim_matrix = cosine_similarity(self.features)
        
        # Diagonale und unteres Dreieck ignorieren (Bild A == Bild A ist uninteressant)
        np.fill_diagonal(sim_matrix, 0)
        duplicates = []
        
        # Wir gehen durch die Matrix
        checked = set()
        for i in range(len(sim_matrix)):
            if i in checked: continue
            
            # Finde alle Indices wo Ähnlichkeit > 0.98 (sehr ähnlich)
            # Cosine Similarity: 1.0 = Identisch, 0.0 = Gegenteil
            matches = np.where(sim_matrix[i] > (1 - threshold))[0]
            
            if len(matches) > 0:
                group = [self.paths_sorted[i]] + [self.paths_sorted[m] for m in matches]
                duplicates.append(group)
                checked.update(matches)
                checked.add(i)
        
        print(f"--- GEFUNDENE DUPLIKAT-GRUPPEN: {len(duplicates)} ---")
        dup_dir = os.path.join(OUTPUT_BASE, "Duplicates_Found")
        if os.path.exists(dup_dir): shutil.rmtree(dup_dir)
        
        for idx, group in enumerate(duplicates):
            group_dir = os.path.join(dup_dir, f"Group_{idx}")
            os.makedirs(group_dir, exist_ok=True)
            for p in group:
                shutil.copy2(p, group_dir)
                
        if len(duplicates) == 0:
            print("Keine Duplikate gefunden. Deine Sammlung ist sauber!")
        else:
            print(f"Duplikate wurden zur Überprüfung kopiert nach: {dup_dir}")

    def generate_3d_universe(self):
        """Erstellt eine interaktive HTML-Datei mit 3D-Plot"""
        self.extract_visual_features()
        
        print(">>> Berechne 3D-Koordinaten (PCA)...")
        # Wir reduzieren 2048 Features auf 3 (X, Y, Z)
        pca = PCA(n_components=3)
        components = pca.fit_transform(self.features)
        
        # Daten für Plotly vorbereiten
        df = pd.DataFrame(components, columns=['x', 'y', 'z'])
        df['filepath'] = [os.path.basename(p) for p in self.paths_sorted]
        
        # Optional: Wir machen noch Clustering für Farben
        kmeans = KMeans(n_clusters=8)
        df['cluster'] = kmeans.fit_predict(self.features).astype(str)
        
        print(">>> Generiere HTML...")
        fig = px.scatter_3d(
            df, x='x', y='y', z='z',
            color='cluster',
            hover_name='filepath',
            title='Mein KI Bild-Universum (Interaktiv)',
            opacity=0.7
        )
        
        # Kleines Styling
        fig.update_traces(marker=dict(size=4))
        output_file = os.path.join(OUTPUT_BASE, "3d_universe.html")
        os.makedirs(OUTPUT_BASE, exist_ok=True)
        fig.write_html(output_file)
        
        print(f"--- WOW! ---")
        print(f"Öffne jetzt diese Datei in deinem Browser: {output_file}")

# --- HAUPTPROGRAMM (MENÜ) ---
if __name__ == "__main__":
    print("--- ULTIMATE AI IMAGE TOOL V1.0 ---")
    ai = AI_Mastermind(INPUT_DIR)
    
    while True:
        print("\nWas möchtest du tun?")
        print("[1] Aufräumen: Duplikate finden")
        print("[2] Sortieren: Nach meinen Regeln (Natur, Essen, ...)")
        print("[3] Visualisieren: 3D Universum erstellen")
        print("[4] Beenden")
        
        choice = input("Wahl: ")
        
        if choice == "1":
            ai.find_duplicates(threshold=0.02) # 2% Toleranz
        
        elif choice == "2":
            print("\nGib deine Kategorien ein (getrennt durch Komma).")
            print("Beispiel: Urlaub, Rechnung, Auto, Haustier")
            user_cats = input("Kategorien: ")
            cat_list = [c.strip() for c in user_cats.split(",")]
            if len(cat_list) > 1:
                ai.sort_by_text_categories(cat_list)
            else:
                print("Zu wenige Kategorien!")
                
        elif choice == "3":
            ai.generate_3d_universe()
            
        elif choice == "4":
            print("Bye!")
            break
        else:
            print("Ungültige Eingabe.")

C:\Users\matth\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


--- ULTIMATE AI IMAGE TOOL V1.0 ---
[68094 Bilder geladen]

Was möchtest du tun?
[1] Aufräumen: Duplikate finden
[2] Sortieren: Nach meinen Regeln (Natur, Essen, ...)
[3] Visualisieren: 3D Universum erstellen
[4] Beenden

Gib deine Kategorien ein (getrennt durch Komma).
Beispiel: Urlaub, Rechnung, Auto, Haustier
>>> Lade CLIP (für Text-Sortierung)...


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


>>> Sortiere nach: ['Urlaub', 'Nahrung', 'Natur', 'Personen', 'Tiere', 'Wasser', 'Auto', 'Plüschtier', 'Elektronik', 'Landschaft', 'Sonstiges']


100%|██████████| 2128/2128 [14:14<00:00,  2.49it/s]

Fertig! Ergebnisse in ../DSAI_Ultimate_Output\Sorted_by_Text

Was möchtest du tun?
[1] Aufräumen: Duplikate finden
[2] Sortieren: Nach meinen Regeln (Natur, Essen, ...)
[3] Visualisieren: 3D Universum erstellen
[4] Beenden


>>> Lade ResNet50 (für Struktur & Duplikate)...
>>> Scanne Bilder (ResNet)...


100%|██████████| 2128/2128 [01:47<00:00, 19.84it/s] 


>>> Features fertig: (22477, 2048)
>>> Berechne 3D-Koordinaten (PCA)...


ValueError: Length of values (68094) does not match length of index (22477)

Hier ist die detaillierte Erklärung des Codes, formatiert als Markdown, damit du sie direkt in dein Jupyter Notebook oder deine Dokumentation kopieren kannst.

---

# 🧠 Der SmartImageSorter – Code-Erklärung

Dieses Skript nutzt **Unsupervised Learning** (unüberwachtes Lernen). Das bedeutet, die KI weiß nicht, was auf den Bildern ist (keine Labels wie "Hund" oder "Katze"). Stattdessen analysiert sie die visuelle Struktur und gruppiert ähnliche Bilder selbstständig.

## 1. Die Grundbausteine (Konzepte)

Bevor wir in den Code schauen, hier die verwendeten Technologien:

* **ResNet18 (Feature Extractor):** Ein vor-trainiertes neuronales Netz, das eigentlich für Bildklassifizierung (ImageNet) gemacht ist. Wir nutzen es als "Auge", um Bilder in abstrakte Zahlenreihen umzuwandeln.
* **PCA (Principal Component Analysis):** Ein statistisches Verfahren, um Daten zu komprimieren. Es entfernt Rauschen und behält nur die wichtigen Unterschiede zwischen Bildern.
* **K-Means:** Ein Algorithmus, der Datenpunkte in  Gruppen (Cluster) sortiert, indem er immer wieder die Mittelpunkte (Zentroiden) der Gruppen verschiebt.
* **t-SNE:** Ein Algorithmus zur Visualisierung. Er presst hoch-dimensionale Daten (512 Dimensionen) auf 2 Dimensionen () zusammen, damit wir sie als Karte plotten können.

## 2. Der Code im Detail

Der Code ist in einer Klasse `SmartImageSorter` organisiert. Hier ist die Aufschlüsselung der einzelnen Funktionen:

### A. Datenvorbereitung (`RobustImageDataset` & `__init__`)

Das Ziel hier ist **Fehlertoleranz**. Normale Skripte stürzen ab, wenn eine Datei beschädigt ist oder ein falsches Format hat.

```python
class RobustImageDataset(Dataset):
    # ...
    def __getitem__(self, idx):
        try:
            # Versucht Bild zu öffnen und in RGB zu wandeln
            with Image.open(path) as img:
                img = img.convert("RGB")
                # ...
        except:
            # Wenn Datei kaputt: Gib "None" zurück (wird später ignoriert)
            return None, path

```

* **Warum RGB?** Manche Bilder sind Schwarz-Weiß (1 Kanal), manche haben Transparenz (4 Kanäle). ResNet erwartet strikt 3 Kanäle (Rot, Grün, Blau). `.convert("RGB")` erzwingt das.
* **`clean_collate`**: Eine Hilfsfunktion, die defekte Bilder (`None`) aus dem Daten-Batch filtert, bevor sie in die KI kommen.

### B. Das "Sehen" (`extract_features`)

Hier passiert die Magie der Umwandlung von Pixeln in Verständnis (Feature Extraction).

```python
def _load_model(self):
    self.model = models.resnet18(weights='DEFAULT')
    self.model.fc = nn.Identity()  # <--- WICHTIG!

```

* **`model.fc = nn.Identity()`**: Das Standard-ResNet hat am Ende eine Schicht ("Fully Connected"), die Klassifizierungen vornimmt (z.B. "Das ist zu 90% ein Hund"). Wir löschen diese Schicht. Jetzt gibt das Netz stattdessen den **Feature-Vektor** (512 Zahlen) aus, der das Bild beschreibt (Formen, Farben, Texturen).

```python
def extract_features(self):
    # ...
    preds = self.model(imgs) # Ergebnis: [Batch_Size, 512]
    # ...

```

* Wir schieben alle Bilder durch das Netz und speichern diese 512-Zahlen-Pakete in einer großen Liste (`self.features`).

### C. Das "Denken" (`analyze_and_cluster`)

Jetzt haben wir tausende Vektoren mit je 512 Zahlen. Das ist zu viel für einfache Algorithmen und enthält oft Rauschen.

```python
def analyze_and_cluster(self):
    # Schritt 1: PCA (Rauschunterdrückung)
    pca = PCA(n_components=0.95)
    self.pca_features = pca.fit_transform(self.features)

```

* **PCA (`n_components=0.95`):** Wir sagen: "Behalte 95% der Informationen". PCA reduziert die 512 Dimensionen oft auf ca. 50-100. Das entfernt unwichtige Details (z.B. Hintergrundrauschen) und macht das folgende Clustering präziser.

```python
    # Schritt 2: K-Means (Gruppierung)
    self.kmeans = KMeans(n_clusters=self.n_clusters, ...)
    self.labels = self.kmeans.fit_predict(self.pca_features)

```

* **K-Means:** Der Algorithmus schaut sich die reduzierten Daten an und findet `N_CLUSTERS` Gruppen. Jedes Bild bekommt ein Label (z.B. `0`, `1`, `2`, `3`, `4`).

### D. Die Anomalie-Suche (`find_anomalies`)

Ein Highlight für die Qualitätskontrolle oder zum Finden von "besonderen" Bildern.

```python
def find_anomalies(self, n_anomalies=5):
    all_dists = self.kmeans.transform(self.pca_features)
    min_dists = np.min(all_dists, axis=1) # Distanz zum eigenen Cluster-Zentrum

```

* **Logik:** Ein "normales" Bild liegt mathematisch nah am Zentrum seiner Gruppe. Ein "seltsames" Bild liegt weit weg.
* Wir sortieren die Bilder einfach nach dieser Distanz. Die Bilder mit der größten Distanz sind die **Anomalien** (Ausreißer).

### E. Die Visualisierung (`visualize_map`)

```python
def visualize_map(self):
    tsne = TSNE(n_components=2, ...)
    vis_features = tsne.fit_transform(self.pca_features)

```

* **t-SNE:** Da wir 4D- oder 100D-Daten nicht zeichnen können, quetscht t-SNE die Daten auf 2D zusammen. Dabei versucht es, **Nachbarschaften zu erhalten**: Was im hohen Raum nah beieinander war, soll auch auf dem Blatt Papier nah beieinander sein.
* Das Ergebnis wird als `ki_analyse_map.png` gespeichert.

## 3. Zusammenfassung des Workflows

1. **Input:** Ordner mit chaotischen Bildern (`.jpg`, `.png`, etc.).
2. **Preprocessing:** Bilder auf  Pixel skalieren und normalisieren.
3. **Feature Extraction:** ResNet18 macht aus jedem Bild einen Vektor (512 Zahlen).
4. **Dimension Reduction:** PCA dampft die 512 Zahlen auf das Wesentliche ein.
5. **Clustering:** K-Means teilt die Bilder basierend auf den PCA-Daten in Gruppen ein.
6. **Action:**
* Anomalien werden identifiziert.
* Bilder werden physisch in Ordner verschoben (`Cluster_0`, `Cluster_1`, ...).
* Eine visuelle Karte wird gezeichnet.



## 4. Warum ist dieser Code "besser"?

* **Effizienz:** Durch **PCA** vor dem Clustering und **t-SNE** spart der Code Rechenzeit und liefert sauberere Ergebnisse, als wenn man K-Means direkt auf die Rohdaten loslassen würde.
* **Stabilität:** Durch das `RobustImageDataset` bricht der Prozess nicht ab, nur weil ein Bild mal defekt ist.
* **Modularität:** Das Modell kann leicht ausgetauscht werden (z.B. `resnet50` statt `resnet18` für noch mehr Details), ohne den gesamten Code neu schreiben zu müssen.